### Imports

In [ ]:
from pathlib import Path
import sys

In [ ]:

root_dir = Path.cwd().resolve().parents[0]
sys.path.insert(0, str(root_dir / "src"))
root_dir


In [ ]:
from deal_hunter.config import settings

In [ ]:
settings.rss_feed_url

### Testing deals.py , pydantic models

In [ ]:
from deal_hunter.models import ScrapedDeal, Deal,DealSelection, Opportunity

In [ ]:
#making class
sd = ScrapedDeal(
    title = "Test Product" * 20,
    summary = "test SUmmary",
    url = "https://example.com",
    details = "some details" * 100,
    features = " Feature List" * 100
)



In [ ]:
#checking methods
sd.truncate()
print(repr(sd))
print(sd.describe())
print(len(sd.title))
print(len(sd.details))

In [ ]:
# Verifying other models
d = Deal(product_description="A widget", price=29.99, url="https://example.com")
ds = DealSelection(deals=[d])
o = Opportunity(deal=d, estimate=50.0, discount=0.40)
print(o)

In [ ]:
from deal_hunter.services import Rss_Service

rss = Rss_Service()
deals = rss.scrape_feeds(
    feed_urls=settings.rss_feed_url,
    max_per_feed=3,
)
print(f"Scraped {len(deals)} deals")
for d in deals[:3]:
    print(d.describe())
    print()

In [ ]:
already_seen = {d.url for d in deals}
print(f"Known URLs: {len(already_seen)}")

deals_round2 = rss.scrape_feeds(
    feed_urls=settings.rss_feed_url,
    known_urls=already_seen,
    max_per_feed=3,
)
print(f"Round 2: {len(deals_round2)} new deals (should be 0 or very few)")

### testing notification service

In [ ]:
from deal_hunter.config import settings
from deal_hunter.services.notifications import PushoverNotifier

notifier = PushoverNotifier(
    user_key=settings.pushover_user,
    token=settings.pushover_token,
    url=settings.pushover_url,
)

print("user set?", bool(settings.pushover_user))
print("token set?", bool(settings.pushover_token))
print("url:", settings.pushover_url)

In [ ]:
ok = notifier.send("Phase 4 test from scanning.ipynb")
print("send() returned:", ok)

In [ ]:
from deal_hunter.config import settings

print("user set?", bool(settings.pushover_user), "len:", len(settings.pushover_user))
print("token set?", bool(settings.pushover_token), "len:", len(settings.pushover_token))
print("url:", settings.pushover_url)


In [ ]:
import requests
from deal_hunter.config import settings

payload = {
    "user": settings.pushover_user,
    "token": settings.pushover_token,
    "message": "debug test from notebook",
    "sound": "cashregister",
}

resp = requests.post(settings.pushover_url, data=payload, timeout=10)
print("HTTP status:", resp.status_code)
print("Response text:", resp.text)

### testing scanner agent

In [ ]:
from deal_hunter.agents.scanner import ScannerAgent

In [ ]:
agent = ScannerAgent()
result = agent.test_scan()
print(result)

In [ ]:
### Testing MessagingAgent

Same flow as week8/day3 cells 21-22: import, push a test string, then try `notify` with a fake deal.

In [ ]:
from deal_hunter.agents import MessagingAgent

messenger = MessagingAgent()
print("notifier type:", type(messenger.notifier).__name__)
print("model:", messenger.model)

In [ ]:
messenger.push("Phase 6 test from scanning.ipynb")

In [ ]:
messenger.notify(
    "Samsung 60-inch LED TV, 4K UHD, smart platform with built-in apps",
    300,
    1000,
    "https://www.samsung.com/tv",
)

### Full pipeline: scan -> notify

Wire the ScannerAgent output into MessagingAgent.
The scanner picks the best deals from RSS, then the messenger crafts an LLM-written notification for each one and pushes it.

In [ ]:
from deal_hunter.agents import ScannerAgent, MessagingAgent

scanner = ScannerAgent()
messenger = MessagingAgent()

result = scanner.scan()

if result is None:
    print("No deals found, nothing to notify about.")
else:
    print(f"Scanner returned {len(result.deals)} deals\n")
    for deal in result.deals:
        print(f"  {deal.product_description[:60]}...  ${deal.price}")
        print(f"  {deal.url}\n")

In [ ]:
if result and result.deals:
    deal = result.deals[0]
    ok = messenger.notify(
        description=deal.product_description,
        deal_price=deal.price,
        estimated_true_value=deal.price * 2,
        url=deal.url,
    )
    print(f"Push sent: {ok}")
else:
    print("No deals to notify about.")

### Using test_scan data (no network needed)

If you want to run the pipeline without hitting RSS or OpenAI, use `test_scan()` which returns hardcoded deals.

In [ ]:
test_result = scanner.test_scan()

for deal in test_result.deals:
    ok = messenger.notify(
        description=deal.product_description,
        deal_price=deal.price,
        estimated_true_value=deal.price * 2,
        url=deal.url,
    )
    print(f"${deal.price:>7.2f} | pushed={ok} | {deal.product_description[:50]}...")